In [33]:

import gurobipy as gp
from gurobipy import GRB
import numpy as np


In [7]:
def solve_lower_level(x_fixed, verbose=False):
    """
    Solve the x-parameterized binary follower problem
    for Exercise 9.15.

    Parameters
    ----------
    x_fixed : list or tuple of length 3
        Fixed leader decision (x1, x2, x3).

    verbose : bool
        Whether to display Gurobi solver output.

    Returns

    phi : float
        Optimal follower objective value.

    y_opt : list
        Optimal binary follower solution.
    """

    if len(x_fixed) != 3:
        raise ValueError("x_fixed must contain 3 values.")

    ll = gp.Model("Lower_Level")
    ll.Params.OutputFlag = int(verbose)

    # Follower variables remain BINARY
    y = ll.addVars(3, vtype=GRB.BINARY, name="y")

    ll.setObjective(4*y[0] + 3*y[1] + 3*y[2], GRB.MAXIMIZE)

    ll.addConstr(4*y[0] + 3*y[1] + 2*y[2] <= 4, name="capacity")
    for i in range(3):
        ll.addConstr(y[i] <= 1 - x_fixed[i], name=f"interdiction_{i+1}")

    ll.optimize()
    if ll.Status != GRB.OPTIMAL:
        raise RuntimeError(f"Lower-level solve failed: {ll.Status}")

    phi = ll.ObjVal
    y_opt = [round(y[i].X) for i in range(3)]

    return phi, y_opt


In [ ]:

def solve_refinement(x_fixed, phi, verbose=False):
    """
    Solve the x-parameterized refinement problem
    for Exercise 9.15.

    Parameters
    ----------
    x_fixed : list or tuple
        Integer leader solution x^k.

    phi : float
        Optimal follower value phi(x^k).

    Returns
    -------
    obj : float
        Refined leader objective.

    x_opt, y_opt : list
        Bilevel-feasible solution.
    """

    ref = gp.Model("Refinement")
    ref.Params.OutputFlag = int(verbose)

    x = ref.addVars(3, vtype=GRB.BINARY, name="x")
    y = ref.addVars(3, vtype=GRB.BINARY, name="y")

    profit = 4*y[0] + 3*y[1] + 3*y[2]

    # Leader objective
    ref.setObjective(profit, GRB.MINIMIZE)

    # Leader budget
    ref.addConstr(2*x[0] + x[1] + x[2] <= 2,name="budget")

    # Follower capacity
    ref.addConstr(4*y[0] + 3*y[1] + 2*y[2] <= 4,name="capacity")

    # Follower interdiction constraints
    for i in range(3):
        ref.addConstr(x[i] + y[i] <= 1,name=f"interdiction_{i+1}")

    # Fix all linking variables at x^k
    for i in range(3):
        ref.addConstr(x[i] == round(x_fixed[i]),name=f"fix_x_{i+1}")

    # Enforce follower optimality
    ref.addConstr(profit >= phi, name="follower_optimality")

    ref.optimize()

    if ref.Status == GRB.INFEASIBLE:
        return None

    if ref.Status != GRB.OPTIMAL:
        raise RuntimeError(
            f"Refinement solve failed: {ref.Status}"
        )

    return (
        ref.ObjVal,
        [round(x[i].X) for i in range(3)],
        [round(y[i].X) for i in range(3)]
    )


def get_binding_constraints(model, tol=1e-7):
    if model.Status != GRB.OPTIMAL:
        raise ValueError("Solve the node LP first.")

    variables = model.getVars()
    index = {v.VarName: j for j, v in enumerate(variables)}
    z = np.array([v.X for v in variables])

    binding = []

    # Explicit linear constraints
    for c in model.getConstrs():
        if abs(c.Slack) <= tol:
            row = model.getRow(c)
            coeffs = np.zeros(len(variables))

            for j in range(row.size()):
                v = row.getVar(j)
                coeffs[index[v.VarName]] += row.getCoeff(j)

            # Standardize all inequalities to >=
            if c.Sense == "<":
                coeffs = -coeffs
                rhs = -c.RHS
            elif c.Sense == ">":
                rhs = c.RHS
            else:
                raise ValueError(
                    "Equality constraints need separate handling."
                )

            binding.append((c.ConstrName, coeffs, rhs))

    # Variable bounds (also potentially binding)
    for j, v in enumerate(variables):
        unit = np.zeros(len(variables))
        unit[j] = 1

        if v.LB > -GRB.INFINITY and abs(v.X - v.LB) <= tol:
            binding.append((
                f"{v.VarName}_LB", unit.copy(), v.LB
            ))

        if v.UB < GRB.INFINITY and abs(v.X - v.UB) <= tol:
            binding.append((
                f"{v.VarName}_UB", -unit.copy(), -v.UB
            ))

    return binding



def generate_prop97_cut(model, cut_number, tol=1e-7):
    """
    Generate Proposition 9.7 cut at an integer,
    bilevel-infeasible basic feasible LP solution.

    Assumes:
      - all original variables are integer-constrained
      - all model inequalities have integer data
      - caller verified bilevel infeasibility
    """

    if model.Status != GRB.OPTIMAL:
        raise ValueError("Node LP must be solved first.")

    variables = model.getVars()
    z = np.array([v.X for v in variables])

    # Check integrality
    if np.max(np.abs(z - np.round(z))) > tol:
        raise ValueError("Fractional candidate: branch instead.")

    binding = get_binding_constraints(model, tol)

    if not binding:
        raise ValueError("No binding constraints found.")

    active_matrix = np.array([a for _, a, _ in binding])

    # Check basic feasibility
    if np.linalg.matrix_rank(active_matrix) < len(variables):
        raise ValueError("Candidate is not a basic feasible solution.")

    # Compute alpha, beta, gamma simultaneously
    coefficients = np.sum(active_matrix, axis=0)
    rhs = sum(b for _, _, b in binding)

    # Proposition 9.7 requires integer coefficients
    if (
        np.max(np.abs(coefficients - np.round(coefficients))) > tol
        or abs(rhs - round(rhs)) > tol
    ):
        raise ValueError("Cut data are not integer-valued.")

    coefficients = np.round(coefficients).astype(int)
    rhs = int(round(rhs)) + 1

    # Verify candidate is separated
    if coefficients @ z >= rhs - tol:
        raise ValueError("Cut does not separate the candidate.")

    expression = gp.quicksum(
        int(coefficients[j]) * variables[j]
        for j in range(len(variables))
    )

    cut = model.addConstr(
        expression >= rhs,
        name=f"Prop97_Cut_{cut_number}"
    )

    model.update()

    return cut, coefficients, rhs


Node 0

In [103]:

# Root-node LP relaxation
m = gp.Model("Root_LP")

# All variables continuous at the root node
x = m.addVars(3, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="x")
y = m.addVars(3, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="y")

# Upper-level objective
m.setObjective(
    4*y[0] + 3*y[1] + 3*y[2],
    GRB.MINIMIZE
)

# Upper-level constraint
m.addConstr(2*x[0] + x[1] + x[2] <= 2, name="budget")

# Lower-level feasibility constraints
m.addConstr(4*y[0] + 3*y[1] + 2*y[2] <= 4, name="capacity")
m.addConstr(x[0] + x[1] + x[2] + y[0] + y[1] + y[2] >= 1, name="cut1")
m.addConstr(-3*x[0] + x[1] + x[2] + y[0] + 2*y[1] + 2*y[2] >= -2, name="cut2")
for i in range(3):
    m.addConstr(x[i] + y[i] <= 1, name=f"interdiction_{i+1}")

# Solve LP
m.optimize()

if m.Status == GRB.OPTIMAL:
    print("Root LP objective:", m.ObjVal)
    print("x =", [x[i].X for i in range(3)])
    print("y =", [y[i].X for i in range(3)])
    print("Objective value:", m.ObjVal)

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 7 rows, 6 columns and 24 nonzeros (Min)
Model fingerprint: 0xe62992fb
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]

Presolve time: 0.00s
Presolved: 7 rows, 6 columns, 24 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   1.000000e+00   0.000000e+00      0s
       2    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.01 seconds (0.00 work units)
Optimal objective  0.000000000e+00
Root LP objective: 0.0
x = [0.75, 0.25, 0.0]
y = [0.0, 0.0, 0.0]
Objective value: 0.0


Node 1

In [65]:

# Node 1
m = gp.Model("Node1_LP")

# All variables continuous at the root node
x = m.addVars(3, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="x")
y = m.addVars(3, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="y")

# Upper-level objective
m.setObjective(
    4*y[0] + 3*y[1] + 3*y[2],
    GRB.MINIMIZE
)

# Upper-level constraint
m.addConstr(2*x[0] + x[1] + x[2] <= 2, name="budget")

# Lower-level feasibility constraints
m.addConstr(4*y[0] + 3*y[1] + 2*y[2] <= 4, name="capacity")
m.addConstr(x[0] + x[1] + x[2] + y[0] + y[1] + y[2] >= 1, name="cut1")
m.addConstr(-3*x[0] + x[1] + x[2] + y[0] + 2*y[1] + 2*y[2] >= -2, name="cut2")
m.addConstr(x[0] <= 0, name="branch_x0_0")
m.addConstr(-2*x[0] -3*x[1] -3*x[2] + y[0] >= -5, name="cut3")
m.addConstr(x[0] - x[1] + 2*x[2]+ 2*y[0] + y[1] + 2*y[2] >= 0,name="cut4")
m.addConstr(x[0] + 2*x[1] - x[2]+ 2*y[0] + 2*y[1] + y[2] >= 0,name="cut5")
for i in range(3):
    m.addConstr(x[i] + y[i] <= 1, name=f"interdiction_{i+1}")

# Solve LP
m.optimize()

if m.Status == GRB.OPTIMAL:
    print("Root LP objective:", m.ObjVal)
    print("x =", [x[i].X for i in range(3)])
    print("y =", [y[i].X for i in range(3)])
    print("Objective value:", m.ObjVal)

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 11 rows, 6 columns and 41 nonzeros (Min)
Model fingerprint: 0x518cbabe
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 4 rows and 1 columns
Presolve time: 0.00s
Presolved: 7 rows, 5 columns, 25 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   1.000000e+00   0.000000e+00      0s
       2    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.00 seconds (0.00 work units)
Optimal objective  0.000000000e+00
Root LP objective: 0.0
x = [0.0, 0.6666666666666666, 0.3333333333333333

In [68]:
# Node 3: x[1] <= 0
node3 = m.copy()
node3.addConstr(node3.getVarByName("x[1]") <= 0,name="branch_x1_0")

node3.optimize()

if node3.Status == GRB.OPTIMAL:
    print("Node 3 LP objective:", node3.ObjVal)
    print("x =", [
        node3.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node3.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 12 rows, 6 columns and 42 nonzeros (Min)
Model fingerprint: 0xe475281f
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 8 rows and 2 columns
Presolve time: 0.00s
Presolved: 4 rows, 4 columns, 13 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   1.000000e+00   0.000000e+00      0s
       2    1.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.000000000e+00
Node 3 LP objective: 1.0
x = [0.0, 0.0, 0.6666666666666666]
y = [0.0, 0

In [74]:
# Node 5: x[2] <= 0
node5 = node3.copy()

node5.addConstr(
    node5.getVarByName("x[2]") <= 0,
    name="branch_x2_0"
)

node5.optimize()

if node5.Status == GRB.OPTIMAL:
    print("Node 5 LP objective:", node5.ObjVal)
    print("x =", [
        node5.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node5.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 13 rows, 6 columns and 43 nonzeros (Min)
Model fingerprint: 0x8fc88b86
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 11 rows and 3 columns
Presolve time: 0.00s
Presolved: 2 rows, 3 columns, 6 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   1.000000e+00   0.000000e+00      0s
       1    3.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.01 seconds (0.00 work units)
Optimal objective  3.000000000e+00
Node 5 LP objective: 3.0
x = [0.0, 0.0, 0.0]
y = [0.0, 1.0, 0.0]


In [80]:
node6 = node3.copy()

node6.addConstr(
    node6.getVarByName("x[2]") >= 1,
    name="branch_x2_1"
)

node6.optimize()

if node6.Status == GRB.OPTIMAL:
    print("Node 6 LP objective:", node6.ObjVal)
    print("x =", [
        node6.getVarByName(f"x[{i}]").X for i in range(3)
    ])
    print("y =", [
        node6.getVarByName(f"y[{i}]").X for i in range(3)
    ])

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 13 rows, 6 columns and 43 nonzeros (Min)
Model fingerprint: 0xc632ea2b
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 11 rows and 4 columns
Presolve time: 0.00s
Presolved: 2 rows, 2 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   5.000000e-01   0.000000e+00      0s
       1    1.5000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.500000000e+00
Node 6 LP objective: 1.5
x = [0.0, 0.0, 1.0]
y = [0.0, 0.5, 0.0]


In [83]:

# Branch Node 6 on y[1] = 0.5

node7 = node6.copy()

node7.addConstr(
    node7.getVarByName("y[1]") <= 0,
    name="branch_y1_0"
)

# Right child: Node 8, y[1] >= 1
node8 = node6.copy()

node8.addConstr(
    node8.getVarByName("y[1]") >= 1,
    name="branch_y1_1"
)

node7.optimize()

if node7.Status == GRB.OPTIMAL:
    print("Node 7 LP objective:", node7.ObjVal)

    print("x =", [
        node7.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])

    print("y =", [
        node7.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])

elif node7.Status == GRB.INFEASIBLE:
    print("Node 7 is infeasible")



Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 14 rows, 6 columns and 44 nonzeros (Min)
Model fingerprint: 0x66b5a833
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 14 rows and 6 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  2.000000000e+00
Node 7 LP objective: 2.0
x = [0.0, 0.0, 1.0]
y = [0.5, 0.0, 0.0]


In [84]:

# Node 9: y[0] <= 0
node9 = node7.copy()

node9.addConstr(
    node9.getVarByName("y[0]") <= 0,
    name="branch_y0_0"
)

# Node 10: y[0] >= 1
node10 = node7.copy()

node10.addConstr(
    node10.getVarByName("y[0]") >= 1,
    name="branch_y0_1"
)

node9.optimize()

if node9.Status == GRB.INFEASIBLE:
    print("Node 9: infeasible — fathomed")

elif node9.Status == GRB.OPTIMAL:
    print("Node 9 LP objective:", node9.ObjVal)
    print("x =", [
        node9.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node9.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])



Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 15 rows, 6 columns and 45 nonzeros (Min)
Model fingerprint: 0x3d338611
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 13 rows and 5 columns
Presolve time: 0.00s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Infeasible model
Node 9: infeasible — fathomed


In [85]:
node10.optimize()

if node10.Status == GRB.OPTIMAL:
    print("Node 10 LP objective:", node10.ObjVal)
    print("x =", [
        node10.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node10.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])

elif node10.Status == GRB.INFEASIBLE:
    print("Node 10: infeasible")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 15 rows, 6 columns and 45 nonzeros (Min)
Model fingerprint: 0x943c5c94
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 15 rows and 6 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  4.000000000e+00
Node 10 LP objective: 4.0
x = [0.0, 0.0, 1.0]
y = [1.0, 0.0, 0.0]


In [86]:
node8.optimize()

if node8.Status == GRB.OPTIMAL:
    print("Node 8 LP objective:", node8.ObjVal)
    print("x =", [
        node8.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node8.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])

elif node8.Status == GRB.INFEASIBLE:
    print("Node 8: infeasible")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 14 rows, 6 columns and 44 nonzeros (Min)
Model fingerprint: 0x72149d0f
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 14 rows and 6 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  3.000000000e+00
Node 8 LP objective: 3.0
x = [0.0, 0.0, 1.0]
y = [0.0, 1.0, 0.0]


In [91]:

# Node 4: right child of Node 1
node4 = m.copy()

node4.addConstr(
    node4.getVarByName("x[1]") >= 1,
    name="branch_x1_1"
)

node4.optimize()

if node4.Status == GRB.OPTIMAL:
    print("Node 4 LP objective:", node4.ObjVal)

    print("x =", [
        node4.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])

    print("y =", [
        node4.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])

elif node4.Status == GRB.INFEASIBLE:
    print("Node 4: infeasible")


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 12 rows, 6 columns and 42 nonzeros (Min)
Model fingerprint: 0x10d0336b
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 12 rows and 6 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  0.000000000e+00
Node 4 LP objective: 0.0
x = [0.0, 1.0, 0.5]
y = [0.0, 0.0, 0.0]


In [92]:

# Node 11: x[2] <= 0
node11 = node4.copy()

node11.addConstr(
    node11.getVarByName("x[2]") <= 0,
    name="branch_x2_0"
)

# Node 12: x[2] >= 1
node12 = node4.copy()

node12.addConstr(
    node12.getVarByName("x[2]") >= 1,
    name="branch_x2_1"
)

# Solve Node 11 first (DFS)
node11.optimize()

if node11.Status == GRB.OPTIMAL:
    print("Node 11 LP objective:", node11.ObjVal)
    print("x =", [
        node11.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node11.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])

elif node11.Status == GRB.INFEASIBLE:
    print("Node 11: infeasible")


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 13 rows, 6 columns and 43 nonzeros (Min)
Model fingerprint: 0x6e1a66c1
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 11 rows and 4 columns
Presolve time: 0.00s
Presolved: 2 rows, 2 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   5.000000e-01   0.000000e+00      0s
       1    1.5000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.500000000e+00
Node 11 LP objective: 1.5
x = [0.0, 1.0, 0.0]
y = [0.0, 0.0, 0.5]


In [93]:

# Node 13: y[2] <= 0
node13 = node11.copy()

node13.addConstr(
    node13.getVarByName("y[2]") <= 0,
    name="branch_y2_0"
)

# Node 14: y[2] >= 1
node14 = node11.copy()

node14.addConstr(
    node14.getVarByName("y[2]") >= 1,
    name="branch_y2_1"
)

# Solve Node 13 first (DFS)
node13.optimize()

if node13.Status == GRB.OPTIMAL:
    print("Node 13 LP objective:", node13.ObjVal)
    print("x =", [
        node13.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node13.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])

elif node13.Status == GRB.INFEASIBLE:
    print("Node 13: infeasible")


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 14 rows, 6 columns and 44 nonzeros (Min)
Model fingerprint: 0xab60119d
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 14 rows and 6 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  2.000000000e+00
Node 13 LP objective: 2.0
x = [0.0, 1.0, 0.0]
y = [0.5, 0.0, 0.0]


In [94]:

# Node 15: y1 <= 0
node15 = node13.copy()
node15.addConstr(
    node15.getVarByName("y[0]") <= 0,
    name="branch_y0_0"
)

# Node 16: y1 >= 1
node16 = node13.copy()
node16.addConstr(
    node16.getVarByName("y[0]") >= 1,
    name="branch_y0_1"
)

# Solve Node 15 first (DFS)
node15.optimize()

if node15.Status == GRB.INFEASIBLE:
    print("Node 15: infeasible — fathomed")

elif node15.Status == GRB.OPTIMAL:
    print("Node 15 LP objective:", node15.ObjVal)
    print("x =", [
        node15.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node15.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 15 rows, 6 columns and 45 nonzeros (Min)
Model fingerprint: 0xb1400d46
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 13 rows and 5 columns
Presolve time: 0.00s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Infeasible model
Node 15: infeasible — fathomed


In [95]:

# Solve Node 16
node16.optimize()

if node16.Status == GRB.OPTIMAL:
    print("Node 16 LP objective:", node16.ObjVal)
    print("x =", [
        node16.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node16.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])

elif node16.Status == GRB.INFEASIBLE:
    print("Node 16: infeasible")


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 15 rows, 6 columns and 45 nonzeros (Min)
Model fingerprint: 0xec976ed5
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 15 rows and 6 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  4.000000000e+00
Node 16 LP objective: 4.0
x = [0.0, 1.0, 0.0]
y = [1.0, 0.0, 0.0]


In [100]:

# Node 14: y[2] >= 1
node14 = node11.copy()

node14.addConstr(
    node14.getVarByName("y[2]") >= 1,
    name="branch_y2_1"
)

node14.optimize()

if node14.Status == GRB.OPTIMAL:
    print("Node 14 LP objective:", node14.ObjVal)
    print("x =", [
        node14.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node14.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])

elif node14.Status == GRB.INFEASIBLE:
    print("Node 14: infeasible")


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 14 rows, 6 columns and 44 nonzeros (Min)
Model fingerprint: 0x2777bca2
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 14 rows and 6 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  3.000000000e+00
Node 14 LP objective: 3.0
x = [0.0, 1.0, 0.0]
y = [0.0, 0.0, 1.0]


In [101]:

# Node 12: x[2] >= 1
node12 = node4.copy()

node12.addConstr(
    node12.getVarByName("x[2]") >= 1,
    name="branch_x2_1"
)

node12.optimize()

if node12.Status == GRB.OPTIMAL:
    print("Node 12 LP objective:", node12.ObjVal)
    print("x =", [
        node12.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node12.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])

elif node12.Status == GRB.INFEASIBLE:
    print("Node 12: infeasible — fathomed")


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 13 rows, 6 columns and 43 nonzeros (Min)
Model fingerprint: 0xdb40b8d0
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 13 rows and 6 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  4.000000000e+00
Node 12 LP objective: 4.0
x = [0.0, 1.0, 1.0]
y = [1.0, 0.0, 0.0]


In [104]:

# Node 2: right child of root
node2 = m.copy()

node2.addConstr(
    node2.getVarByName("x[0]") >= 1,
    name="branch_x0_1"
)

node2.optimize()

if node2.Status == GRB.OPTIMAL:
    print("Node 2 LP objective:", node2.ObjVal)
    print("x =", [
        node2.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node2.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])

elif node2.Status == GRB.INFEASIBLE:
    print("Node 2: infeasible")


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 8 rows, 6 columns and 25 nonzeros (Min)
Model fingerprint: 0x3516cb1a
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]

Presolve removed 6 rows and 4 columns
Presolve time: 0.00s
Presolved: 2 rows, 2 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   5.000000e-01   0.000000e+00      0s
       1    1.5000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.500000000e+00
Node 2 LP objective: 1.5
x = [1.0, 0.0, 0.0]
y = [0.0, 0.5, 0.0]


In [105]:

# Node 17: y2 <= 0
node17 = node2.copy()

node17.addConstr(
    node17.getVarByName("y[1]") <= 0,
    name="branch_y1_0"
)

# Node 18: y2 >= 1
node18 = node2.copy()

node18.addConstr(
    node18.getVarByName("y[1]") >= 1,
    name="branch_y1_1"
)

# Solve Node 17 first (DFS)
node17.optimize()

if node17.Status == GRB.OPTIMAL:
    print("Node 17 LP objective:", node17.ObjVal)
    print("x =", [
        node17.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node17.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])

elif node17.Status == GRB.INFEASIBLE:
    print("Node 17: infeasible")


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 9 rows, 6 columns and 26 nonzeros (Min)
Model fingerprint: 0x19e36b58
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]

Presolve removed 9 rows and 6 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.5000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.500000000e+00
Node 17 LP objective: 1.5
x = [1.0, 0.0, 0.0]
y = [0.0, 0.0, 0.5]


In [106]:

# Node 19: y3 <= 0
node19 = node17.copy()

node19.addConstr(
    node19.getVarByName("y[2]") <= 0,
    name="branch_y2_0"
)

# Node 20: y3 >= 1
node20 = node17.copy()

node20.addConstr(
    node20.getVarByName("y[2]") >= 1,
    name="branch_y2_1"
)

# Solve Node 19 first (DFS)
node19.optimize()

if node19.Status == GRB.OPTIMAL:
    print("Node 19 LP objective:", node19.ObjVal)
    print("x =", [
        node19.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node19.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])

elif node19.Status == GRB.INFEASIBLE:
    print("Node 19: infeasible — fathomed")


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 10 rows, 6 columns and 27 nonzeros (Min)
Model fingerprint: 0xca6f0ea8
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]

Presolve removed 8 rows and 5 columns
Presolve time: 0.00s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Infeasible model
Node 19: infeasible — fathomed


In [107]:

node20.optimize()

if node20.Status == GRB.OPTIMAL:
    print("Node 20 LP objective:", node20.ObjVal)
    print("x =", [
        node20.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node20.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])

elif node20.Status == GRB.INFEASIBLE:
    print("Node 20: infeasible")


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 10 rows, 6 columns and 27 nonzeros (Min)
Model fingerprint: 0xb36dded2
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]

Presolve removed 10 rows and 6 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  3.000000000e+00
Node 20 LP objective: 3.0
x = [1.0, 0.0, 0.0]
y = [0.0, 0.0, 1.0]


In [96]:

x_root = [x[i].X for i in range(3)]
y_root = [y[i].X for i in range(3)]

phi, y_follower = solve_lower_level(x_root)

print("Follower optimal value:", phi)
print("Follower optimal solution:", y_follower)

# Check bilevel feasibility
leader_y_value = (4*y_root[0] + 3*y_root[1] + 3*y_root[2])

is_bilevel_feasible = leader_y_value >= phi - 1e-6

print("Candidate follower value:", leader_y_value)
print("Bilevel feasible:", is_bilevel_feasible)


Follower optimal value: 4.0
Follower optimal solution: [1, 0, 0]
Candidate follower value: 0.0
Bilevel feasible: False


In [108]:

node18.optimize()

if node18.Status == GRB.OPTIMAL:
    print("Node 18 LP objective:", node18.ObjVal)
    print("x =", [
        node18.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node18.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])

elif node18.Status == GRB.INFEASIBLE:
    print("Node 18: infeasible — fathomed")


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 9 rows, 6 columns and 26 nonzeros (Min)
Model fingerprint: 0xd9afa7cb
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]

Presolve removed 9 rows and 6 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  3.000000000e+00
Node 18 LP objective: 3.0
x = [1.0, 0.0, 0.0]
y = [0.0, 1.0, 0.0]


In [97]:
result = solve_refinement(x8, phi)
print("Refinement result:", result)


Refinement result: (4.0, [0, 0, 1], [1, 0, 0])


In [98]:

binding = get_binding_constraints(node8)

print("Binding constraints at current node:")
for name, coeffs, rhs in binding:
    print(f"{name}: {coeffs} >= {rhs:g}")

A_active = np.array([row for _, row, _ in binding])

rank = np.linalg.matrix_rank(A_active)
n_variables = len(node8.getVars())

print("\nActive constraint rank:", rank)
print("Number of variables:", n_variables)
print("Basic feasible solution:", rank == n_variables)


Binding constraints at current node:
branch_x0_0: [-1. -0. -0. -0. -0. -0.] >= -0
interdiction_2: [-0. -1. -0. -0. -1. -0.] >= -1
interdiction_3: [-0. -0. -1. -0. -0. -1.] >= -1
branch_x1_0: [-0. -1. -0. -0. -0. -0.] >= -0
branch_x2_1: [0. 0. 1. 0. 0. 0.] >= 1
branch_y1_1: [0. 0. 0. 0. 1. 0.] >= 1
x[0]_LB: [1. 0. 0. 0. 0. 0.] >= 0
x[1]_LB: [0. 1. 0. 0. 0. 0.] >= 0
x[2]_UB: [-0. -0. -1. -0. -0. -0.] >= -1
y[0]_LB: [0. 0. 0. 1. 0. 0.] >= 0
y[1]_UB: [-0. -0. -0. -0. -1. -0.] >= -1
y[2]_LB: [0. 0. 0. 0. 0. 1.] >= 0

Active constraint rank: 6
Number of variables: 6
Basic feasible solution: True


In [ ]:

cut, coefficients, rhs = generate_prop97_cut(
    model=m,
    cut_number=1
)

print("Cut coefficients:", coefficients)
print("Cut RHS:", rhs)

# Re-solve the SAME node
m.optimize()
